In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
klix = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/klix_cda.csv")
avaz = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/avaz_cda.csv")
buka = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/buka_cda.csv")
rtrs = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/rtrs_cda.csv")
stav = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/stav_cda.csv")
radiosarajevo = pd.read_csv("/content/drive/MyDrive/CDA-MLPR data/radiosarajevo_cda.csv")


In [ ]:
avaz

,PORTAL,DATUM,NASLOV,TEKST
0,Dnevni Avaz,04.07.2024,Stevandić tvrdi: Nemanjićko nasljeđe starije o...,Poslanici u Narodnoj skupštini Republike Srpsk...
1,Dnevni Avaz,23.04.2023,Pravo proljetno vrijeme: Očekuju se temperatur...,U većem dijelu BiH danas se očekuje pretežno s...
2,Dnevni Avaz,22.07.2022,Džindić inicirao sastanak sa predstavnicima vl...,U sjedištu Vlade Federacije BiH u Sarajevu dan...
3,Dnevni Avaz,15.08.2024,U Zenici 13. augusta nadmašen apsolutni maksim...,"Na području Federacije Bosne i Hercegovine, to..."
4,Dnevni Avaz,18.10.2021,"Dodik: Zakon ne može biti jači od Ustava, SNSD...",Mi ne uzimamo od BiH ništa što joj je Ustavom ...
...,...,...,...,...
49666,Dnevni Avaz,08.12.2019,Zbog afere s marihuanom sutra će biti podnesen...,"Afera u vezi s plantažom marihuane u Srbiji, s..."
49667,Dnevni Avaz,08.12.2019,Fotografija poruke muža je hit u Srbiji: Evo v...,Moderna tehnologija ima tendenciju da uništi m...
49668,Dnevni Avaz,08.12.2019,"Albaniju uzdrmao novi zemljotres, osjetio se i...",Novi zemljotres registriran je u Albaniji magn...
49669,Dnevni Avaz,07.12.2019,Đilas: Vučić prodaje Srbiju za opstanak na vlasti,"Dragan Đilas, predsjednik Stranke slobode i pr..."


# Filter

In [ ]:
TOPIC_KEYWORDS = {
    "euroatlantske_integracije": [
        "euroatlantske integracije",
        "evropske integracije",
        "europske integracije",
        "eu integracije",
        "evropska unija",
        "europska unija",
        "članstvo u eu",
        "kandidatski status",
        "pregovori sa eu",
        "otvaranje pregovora",
        "nato",
        "savez nato",
        "atlantski savez",
        "map",
        "anp",
        "program reformi",
        "brisel",
        "evropska komisija",
        "europska komisija"
    ],

    "negiranje_genocida": [
        "negiranje genocida",
        "genocid u srebrenici",
        "srebrenički genocid",
        "srebrenica",
        "ratni zločin",
        "ratni zločinac",
        "veličanje ratnih zločinaca",
        "presuda za genocid",
        "haški tribunal",
        "mict",
        "icty",
        "inzkov zakon",
        "zakon o zabrani negiranja genocida",
        "revizija historije",
        "revizija istorije",
        "nije bilo genocida",
        "lažni genocid"
    ],

    "gradjanska_vs_konstitutivni": [
        "građanska država",
        "gradjanska država",
        "konstitutivni narodi",
        "legitimno predstavljanje",
        "majorizacija",
        "jedan čovjek jedan glas",
        "unitarna bih",
        "unitarizacija",
        "federalizacija",
        "treći entitet",
        "dejtonski sporazum",
        "dejtonska bih",
        "sejdić finci",
        "sejdić-finci",
        "presuda ljubić",
        "komšić",
        "hrvatsko pitanje",
        "bošnjačka većina"
    ],

    "izborna_reforma": [
        "izborna reforma",
        "izmjene izbornog zakona",
        "izborni zakon",
        "zakon o izborima",
        "centralna izborna komisija",
        "cik",
        "dom naroda",
        "predsjedništvo bih",
        "hrvatski član predsjedništva",
        "legitimno predstavljanje",
        "schmidt",
        "visoki predstavnik",
        "nametnute izmjene",
        "paket integriteta",
        "elektronsko glasanje",
        "skeneri",
        "izborni prag",
        "hdz",
        "hns",
        "čović"
    ]
}

In [ ]:
import re
import pandas as pd

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower()

def topic_score(row, keywords):
    title = normalize_text(row.get("NASLOV", ""))
    subtitle = normalize_text(row.get("PODNASLOV", ""))
    content = normalize_text(row.get("SADRZAJ", ""))

    score = 0
    matched = []

    for kw in keywords:
        kw_lower = kw.lower()

        if kw_lower in title:
            score += 5
            matched.append(kw)

        if kw_lower in subtitle:
            score += 3
            matched.append(kw)

        if kw_lower in content:
            score += 1
            matched.append(kw)

    return score, list(set(matched))

In [ ]:
for topic, keywords in TOPIC_KEYWORDS.items():
    avaz[f"{topic}_score"] = avaz.apply(lambda row: topic_score(row, keywords)[0], axis=1)
    avaz[f"{topic}_matched"] = avaz.apply(lambda row: topic_score(row, keywords)[1], axis=1)

In [ ]:
topic_score_cols = [f"{topic}_score" for topic in TOPIC_KEYWORDS]

avaz["max_topic_score"] = avaz[topic_score_cols].max(axis=1)

filtered_df = avaz[avaz["max_topic_score"] > 0].copy()

In [ ]:
filtered_df

,PORTAL,DATUM,NASLOV,TEKST,euroatlantske_integracije_score,euroatlantske_integracije_matched,negiranje_genocida_score,negiranje_genocida_matched,gradjanska_vs_konstitutivni_score,gradjanska_vs_konstitutivni_matched,izborna_reforma_score,izborna_reforma_matched,max_topic_score
14,Dnevni Avaz,26.07.2022,Da li će se Komšić pojaviti na otvaranju Pelje...,Večeras će svečano biti otvoren Pelješki most....,0,[],0,[],5,[komšić],0,[],5
16,Dnevni Avaz,27.12.2022,"Kako je DF od ""nećemo s HDZ-om u ZDK"", došao d...",Nakon oktobarskih izbora u Zeničko-dobojskom k...,0,[],0,[],5,[komšić],5,[hdz],5
40,Dnevni Avaz,29.03.2021,Džaferović primio u posjetu ambasadora Rumunij...,Član Predsjedništva BiH Šefik Džaferović primi...,5,[nato],0,[],0,[],0,[],5
86,Dnevni Avaz,03.12.2021,Čović: Spremni smo izbrisati sve prefikse iz u...,U Mostaru je danas održana sjednica Predsjedni...,0,[],0,[],0,[],5,[čović],5
92,Dnevni Avaz,05.12.2024,Za svega pet dana uradili ogroman posao: Donat...,Na lokalitetu Komadinovo vrelo nedaleko od Jab...,5,[nato],0,[],0,[],0,[],5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
49567,Dnevni Avaz,26.12.2019,"Vučić poručio Komšiću: Pustite nas na miru, šu...",Predsjednik Srbije Aleksandar Vučić izjavio je...,0,[],0,[],5,[komšić],0,[],5
49599,Dnevni Avaz,22.12.2019,Vulin nakon Komšićeve izjave o Kosovu: Ne čini...,Ministar odbrane Srbije Aleksandar Vulin oglas...,0,[],0,[],5,[komšić],0,[],5
49614,Dnevni Avaz,20.12.2019,CIA otkrila mapu: Međunarodna granica trebala ...,Ako Beograd i Priština mogu da se dogovore oko...,5,[map],0,[],0,[],0,[],5
49651,Dnevni Avaz,13.12.2019,Ratni zločinac Šešelj brani kontroverznog Hand...,Ratni zločinac Vojislav Šešelj opleo je po dra...,0,[],10,"[ratni zločin, ratni zločinac]",0,[],0,[],10


## Chat gpt

In [ ]:
import re
import math
import difflib
import pandas as pd

# ============================================================
# 1. Spajanje svih portala
# ============================================================

portal_dfs = {
    "klix": klix,
    "avaz": avaz,
    "buka": buka,
    "rtrs": rtrs,
    "stav": stav,
    "radiosarajevo": radiosarajevo,
}

df = avaz

In [ ]:
# ============================================================
# 2. Normalizacija teksta
# ============================================================

CHAR_MAP = str.maketrans({
    "č": "c",
    "ć": "c",
    "đ": "dj",
    "š": "s",
    "ž": "z",
    "Č": "c",
    "Ć": "c",
    "Đ": "dj",
    "Š": "s",
    "Ž": "z",
})

STOPWORDS = {
    "i", "u", "o", "a", "ali", "ili", "pa", "te",
    "sa", "s", "za", "na", "od", "do", "po", "iz", "uz",
    "bez", "kao", "kod", "pri", "prema", "kroz",
    "je", "su", "se", "bi", "bih", "bila", "bilo", "bili",
    "da", "koji", "koja", "koje", "kojih", "ovaj", "ova", "ovo"
}

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).translate(CHAR_MAP).lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def tokenize(x):
    x = normalize_text(x)
    return [t for t in x.split() if t and t not in STOPWORDS]

def prepare_text(x):
    tokens = tokenize(x)
    token_set = set(tokens)

    # prefixi hvataju padeže:
    # izborni / izbornog / izbornom -> izborn
    # zakon / zakona / zakonu -> zakon
    prefix_set = set()
    for t in token_set:
        if len(t) >= 6:
            prefix_set.add(t[:6])
        if len(t) >= 5:
            prefix_set.add(t[:5])

    by_first = {}
    for t in token_set:
        if len(t) >= 4:
            by_first.setdefault(t[0], []).append(t)

    return {
        "text": normalize_text(x),
        "tokens": tokens,
        "token_set": token_set,
        "prefix_set": prefix_set,
        "by_first": by_first,
    }


In [ ]:
# ============================================================
# 3. Fuzzy / overlap matching
# ============================================================

def token_matches(pattern_token, prepared, fuzzy_threshold=0.86):
    """
    Vraca True ako se token iz keyworda pojavljuje u tekstu:
    - egzaktno
    - kroz isti prefix/stem
    - kroz malu typo gresku
    """

    token_set = prepared["token_set"]
    prefix_set = prepared["prefix_set"]
    by_first = prepared["by_first"]

    pt = normalize_text(pattern_token)

    if not pt:
        return False

    # Kratke skracenice ne fuzzy-matchamo jer prave šum:
    # EU, NATO, CIK, HDZ, SDA...
    if len(pt) <= 4:
        return pt in token_set

    # Egzaktno
    if pt in token_set:
        return True

    # Stem/prefix za padeže
    if len(pt) >= 6 and pt[:6] in prefix_set:
        return True

    if len(pt) >= 5 and pt[:5] in prefix_set:
        return True

    # Fuzzy typo matching samo među tokenima koji počinju istim slovom
    candidates = by_first.get(pt[0], [])

    for cand in candidates:
        if abs(len(cand) - len(pt)) <= 2:
            ratio = difflib.SequenceMatcher(None, pt, cand).ratio()
            if ratio >= fuzzy_threshold:
                return True

    return False

def phrase_overlap_match(phrase, prepared, min_overlap=0.75):
    """
    Fraza matcha ako dovoljan broj bitnih riječi iz fraze postoji u tekstu.
    Za fraze od 2 riječi tražimo obje riječi.
    Za fraze od 3+ riječi dopuštamo da jedna riječ fali.
    """

    phrase_tokens = tokenize(phrase)

    if not phrase_tokens:
        return False

    matched = 0

    for pt in phrase_tokens:
        if token_matches(pt, prepared):
            matched += 1

    n = len(phrase_tokens)

    if n <= 2:
        needed = n
    else:
        needed = max(2, math.ceil(n * min_overlap))

    return matched >= needed

### Keywords

In [ ]:
# ============================================================
# 4. Širi keywordi po temama
# ============================================================

TOPIC_PHRASES = {
    "euroatlantske_integracije": [
        # direktno
        "euroatlantske integracije",
        "evroatlantske integracije",
        "euroatlantski put",
        "evroatlantski put",
        "atlantske integracije",
        "evropske integracije",
        "europske integracije",
        "evropski put",
        "europski put",
        "put ka evropskoj uniji",
        "put ka europskoj uniji",
        "put prema eu",
        "put u eu",
        "clanstvo u eu",
        "ulazak u eu",
        "pristupanje eu",
        "pristupanje evropskoj uniji",
        "pristupanje europskoj uniji",

        # EU proces
        "evropska unija",
        "europska unija",
        "unija evropska",
        "kandidatski status",
        "status kandidata",
        "zemlja kandidat",
        "otvaranje pregovora",
        "pristupni pregovori",
        "pregovori sa eu",
        "pregovori s eu",
        "pregovaracki okvir",
        "screening proces",
        "skrining proces",
        "izvjestaj o napretku",
        "paket prosirenja",
        "politika prosirenja",
        "proces prosirenja",
        "stabilizacija i pridruzivanje",
        "sporazum o stabilizaciji",
        "ssp",
        "acquis",
        "pravna stecevina",
        "reformska agenda",
        "reformski proces",
        "evropske reforme",
        "europske reforme",
        "14 prioriteta",
        "cetrnaest prioriteta",
        "misljenje evropske komisije",
        "misljenje europske komisije",
        "mehanizam koordinacije",
        "plan rasta",
        "ipa fondovi",
        "brisel",
        "bruxelles",
        "evropska komisija",
        "europska komisija",
        "evropsko vijece",
        "europsko vijece",
        "vijece eu",
        "delegacija eu",

        # NATO
        "nato",
        "nato savez",
        "sjevernoatlantski savez",
        "sjeveroatlantski savez",
        "atlantski savez",
        "clanstvo u nato",
        "ulazak u nato",
        "put u nato",
        "map",
        "akcioni plan za clanstvo",
        "anp",
        "godisnji nacionalni program",
        "program reformi",
        "komisija za nato integracije",
        "partnerstvo za mir",
        "pfp",
        "odbrambena reforma",
        "vojna neutralnost",
        "neutralnost prema nato",
        "nato integracije",
    ],

    "negiranje_genocida": [
        # direktno
        "negiranje genocida",
        "poricanje genocida",
        "relativizacija genocida",
        "umanjivanje genocida",
        "osporavanje genocida",
        "nije bilo genocida",
        "lazni genocid",
        "navodni genocid",
        "takoreceni genocid",
        "genocid se nije desio",
        "zabrana negiranja genocida",
        "zakon o zabrani negiranja genocida",

        # Srebrenica
        "genocid u srebrenici",
        "srebrenicki genocid",
        "srebrenica genocid",
        "srebrenica",
        "potocari",
        "memorijalni centar srebrenica",
        "memorijalni centar potocari",
        "11 juli",
        "jedanaesti juli",
        "dan sjecanja",
        "rezolucija o srebrenici",
        "rezolucija o genocidu",
        "un rezolucija o srebrenici",
        "generalna skupstina un",

        # ratni zlocini
        "ratni zlocin",
        "ratni zlocini",
        "ratni zlocinac",
        "ratni zlocinci",
        "presudjeni ratni zlocinac",
        "velicanje ratnih zlocinaca",
        "glorifikacija ratnih zlocinaca",
        "slavljenje ratnih zlocinaca",
        "revizija historije",
        "revizija istorije",
        "historijski revizionizam",
        "istorijski revizionizam",
        "zlocin protiv covjecnosti",
        "etnicko ciscenje",

        # pravni kontekst
        "inzkov zakon",
        "valentin inzko",
        "krivicni zakon bih",
        "izmjene krivicnog zakona",
        "ohr odluka",
        "haski tribunal",
        "haški tribunal",
        "haag",
        "hag",
        "icty",
        "mict",
        "medjunarodni krivicni tribunal",
        "medjunarodni sud pravde",
        "msp",
        "icj",
        "presuda za genocid",
        "presuda mladicu",
        "presuda karadzicu",
        "ratko mladic",
        "radovan karadzic",
        "vojska republike srpske",
        "vrs",
    ],

    "gradjanska_vs_konstitutivni": [
        # direktno
        "gradjanska drzava",
        "gradjanski koncept",
        "gradjanski princip",
        "gradjanska bih",
        "konstitutivni narodi",
        "konstitutivnost naroda",
        "princip konstitutivnosti",
        "ravnopravnost naroda",
        "ravnopravnost konstitutivnih naroda",
        "nacionalna ravnopravnost",
        "nacionalni kljuc",
        "etnicki princip",
        "etnicko predstavljanje",
        "nacionalno predstavljanje",
        "legitimno predstavljanje",

        # sukob modela
        "jedan covjek jedan glas",
        "jedan covjek jedan glas",
        "princip jedan covjek jedan glas",
        "majorizacija",
        "preglasavanje",
        "politicko preglasavanje",
        "bosnjacka vecina",
        "hrvatsko pitanje",
        "srpsko pitanje",
        "hrvatski narod u bih",
        "bošnjaci hrvati srbi",
        "bosnjaci hrvati srbi",
        "tri konstitutivna naroda",
        "ostali gradjani",
        "ostali i gradjani",

        # drzavno uredjenje
        "unitarna bih",
        "unitarna drzava",
        "unitarizacija",
        "unitarizam",
        "centralizacija bih",
        "federalizacija",
        "federalna bih",
        "treci entitet",
        "hrvatski entitet",
        "entitetsko glasanje",
        "dejtonski sporazum",
        "dejtonska bih",
        "dayton",
        "ustav bih",
        "ustavne promjene",
        "ustavna reforma",
        "reforma ustava",
        "promjena ustava",

        # institucije i presude
        "dom naroda",
        "vitalni nacionalni interes",
        "vni",
        "klub bosnjaka",
        "klub hrvata",
        "klub srba",
        "sejdic finci",
        "sejdic-finci",
        "presuda sejdic finci",
        "presuda ljubic",
        "slucaj ljubic",
        "presuda zornic",
        "presuda pilav",
        "evropski sud za ljudska prava",
        "europski sud za ljudska prava",
        "strasbourg",
        "strasbur",
        "zeljko komsic",
        "komsic",
        "dragan covic",
        "covic",
    ],

    "izborna_reforma": [
        # direktno
        "izborna reforma",
        "reforma izbornog zakona",
        "reforma izbornog zakonodavstva",
        "izmjene izbornog zakona",
        "izmjene i dopune izbornog zakona",
        "izborni zakon",
        "zakon o izborima",
        "izborno zakonodavstvo",
        "izborni paket",
        "paket izbornih reformi",

        # integritet izbora
        "integritet izbora",
        "izborni integritet",
        "tehnicke izmjene izbornog zakona",
        "tehnicke izmjene",
        "paket integriteta",
        "elektronsko glasanje",
        "elektronicko glasanje",
        "digitalizacija izbora",
        "skeneri",
        "opticki skeneri",
        "biometrija",
        "video nadzor",
        "brojanje glasova",
        "ponovno brojanje",
        "kontrolno brojanje",
        "biracki spisak",
        "centralni biracki spisak",
        "biracka mjesta",
        "biracki odbori",
        "kradja izbora",
        "izborna kradja",
        "izborni inzenjering",
        "izborna manipulacija",

        # institucije
        "centralna izborna komisija",
        "cik",
        "sredisnje izborno povjerenstvo",
        "sip",
        "predsjednistvo bih",
        "clan predsjednistva",
        "hrvatski clan predsjednistva",
        "bosnjacki clan predsjednistva",
        "srpski clan predsjednistva",
        "dom naroda",
        "federalni dom naroda",
        "parlament federacije",
        "delegati u domu naroda",
        "popunjavanje doma naroda",
        "izbor delegata",
        "izborni prag",
        "kompenzacijski mandati",

        # politicki pregovori
        "legitimno predstavljanje",
        "neumski pregovori",
        "pregovori u neumu",
        "elektorski model",
        "indirektan izbor",
        "direktan izbor",
        "model izbora clanova predsjednistva",
        "schmidtove izmjene",
        "nametnute izmjene",
        "nametanje izbornog zakona",
        "visoki predstavnik",
        "christian schmidt",
        "ohr",
        "hdz prijedlog",
        "hns prijedlog",
        "sda prijedlog",
        "snsd prijedlog",
    ],
}

# ============================================================
# 5. Regex/stem patterni za padeže
# ============================================================

TOPIC_REGEX = {
    "euroatlantske_integracije": [
        r"\beuroatlantsk\w*\s+(integracij\w*|put\w*)",
        r"\bevroatlantsk\w*\s+(integracij\w*|put\w*)",
        r"\bevropsk\w*\s+(integracij\w*|put\w*|unij\w*|komisij\w*|vijec\w*)",
        r"\beuropsk\w*\s+(integracij\w*|put\w*|unij\w*|komisij\w*|vijec\w*)",
        r"\bclanstv\w*\s+u\s+(eu|nato)",
        r"\bulaz\w*\s+u\s+(eu|nato)",
        r"\bpristupan\w*\s+(eu|evropsk\w*|europsk\w*)",
        r"\botvaran\w*\s+pregovor\w*",
        r"\bkandidat\w*\s+status\w*",
        r"\bstatus\w*\s+kandidat\w*",
        r"\bprogram\w*\s+reform\w*",
        r"\bakcion\w*\s+plan\w*\s+za\s+clanstv\w*",
        r"\bsjevernoatlantsk\w*\s+savez\w*",
        r"\bnato\s+integracij\w*",
        r"\bvoj\w*\s+neutralnost\w*",
    ],

    "negiranje_genocida": [
        r"\bnegir\w*\s+genocid\w*",
        r"\bporic\w*\s+genocid\w*",
        r"\brelativiz\w*\s+genocid\w*",
        r"\bumanj\w*\s+genocid\w*",
        r"\bosporav\w*\s+genocid\w*",
        r"\bgenocid\w*\s+u\s+srebrenic\w*",
        r"\bsrebrenick\w*\s+genocid\w*",
        r"\bzabran\w*\s+negiran\w*\s+genocid\w*",
        r"\bratn\w*\s+zlocin\w*",
        r"\bvelican\w*\s+ratn\w*\s+zlocin\w*",
        r"\bglorifik\w*\s+ratn\w*\s+zlocin\w*",
        r"\bpresudj\w*\s+ratn\w*\s+zlocin\w*",
        r"\brevizij\w*\s+(historij\w*|istorij\w*)",
        r"\bhask\w*\s+tribunal\w*",
        r"\bpresud\w*\s+za\s+genocid\w*",
        r"\binzkov\w*\s+zakon\w*",
    ],

    "gradjanska_vs_konstitutivni": [
        r"\bgradjansk\w*\s+(drzav\w*|koncept\w*|princip\w*|bih)",
        r"\bkonstitutivn\w*\s+narod\w*",
        r"\bkonstitutivnost\w*\s+narod\w*",
        r"\blegitimn\w*\s+predstavlj\w*",
        r"\bravnopravnost\w*\s+(narod\w*|konstitutivn\w*)",
        r"\bjedan\w*\s+covjek\w*\s+jedan\w*\s+glas\w*",
        r"\bunitarn\w*\s+(bih|drzav\w*)",
        r"\bunitariz\w*",
        r"\bmajoriz\w*",
        r"\bpreglasav\w*",
        r"\bfederaliz\w*",
        r"\btrec\w*\s+entitet\w*",
        r"\bhrvatsk\w*\s+pitanj\w*",
        r"\bbosnjack\w*\s+vecin\w*",
        r"\bdejtonsk\w*\s+(sporazum\w*|bih)",
        r"\bustavn\w*\s+(promjen\w*|reform\w*)",
        r"\bvitaln\w*\s+nacionaln\w*\s+interes\w*",
        r"\bdom\w*\s+narod\w*",
        r"\bsejdic\w*\s+finci\w*",
        r"\bpresud\w*\s+ljubic\w*",
    ],

    "izborna_reforma": [
        r"\bizborn\w*\s+reform\w*",
        r"\breform\w*\s+izborn\w*\s+zakon\w*",
        r"\bizmjen\w*\s+(i\s+dopun\w*\s+)?izborn\w*\s+zakon\w*",
        r"\bizborn\w*\s+zakon\w*",
        r"\bizborn\w*\s+zakonodavstv\w*",
        r"\bintegritet\w*\s+izbor\w*",
        r"\btehnick\w*\s+izmjen\w*",
        r"\bpaket\w*\s+integritet\w*",
        r"\belektronsk\w*\s+glasan\w*",
        r"\belektronick\w*\s+glasan\w*",
        r"\bdigitalizacij\w*\s+izbor\w*",
        r"\boptic\w*\s+skener\w*",
        r"\bbirack\w*\s+spis\w*",
        r"\bbirack\w*\s+mjest\w*",
        r"\bbirack\w*\s+odbor\w*",
        r"\bcentraln\w*\s+izborn\w*\s+komisij\w*",
        r"\bsredisnj\w*\s+izborn\w*\s+povjerenstv\w*",
        r"\bhrvatsk\w*\s+clan\w*\s+predsjednistv\w*",
        r"\bclan\w*\s+predsjednistv\w*",
        r"\bdelegat\w*\s+u\s+dom\w*\s+narod\w*",
        r"\bpopunjav\w*\s+dom\w*\s+narod\w*",
        r"\bizborn\w*\s+prag\w*",
        r"\bschmidt\w*\s+izmjen\w*",
        r"\bnametn\w*\s+izmjen\w*",
        r"\bnametan\w*\s+izborn\w*\s+zakon\w*",
        r"\bneumsk\w*\s+pregovor\w*",
        r"\bpregovor\w*\s+u\s+neum\w*",
    ],
}


## Nastavak

In [ ]:
# ============================================================
# 6. Scoring
# ============================================================

TEXT_COLUMNS_WEIGHTS = {
    "NASLOV": 5,
    "NADNASLOV": 4,
    "PODNASLOV": 3,
    "RUBRIKA": 2,
    "SADRZAJ": 1,
    "TEKST": 1,       # ako neki portal ima TEKST umjesto SADRZAJ
    "CONTENT": 1,
    "text": 1,
}

def topic_score(row, topic):
    phrases = TOPIC_PHRASES[topic]
    regexes = TOPIC_REGEX[topic]

    total_score = 0
    matched = []

    for col, weight in TEXT_COLUMNS_WEIGHTS.items():
        if col not in row.index:
            continue

        raw_text = row.get(col, "")
        if pd.isna(raw_text) or str(raw_text).strip() == "":
            continue

        prepared = prepare_text(raw_text)
        norm_text = prepared["text"]

        # Regex/stem match
        for pat in regexes:
            if re.search(pat, norm_text):
                total_score += weight * 2
                matched.append(f"{col}:REGEX:{pat}")

        # Phrase overlap/fuzzy match
        for phrase in phrases:
            if phrase_overlap_match(phrase, prepared):
                total_score += weight
                matched.append(f"{col}:{phrase}")

    return total_score, sorted(set(matched))

# Izračunaj za sve teme
for topic in TOPIC_PHRASES:
    results = df.apply(lambda row: topic_score(row, topic), axis=1)
    df[f"{topic}_score"] = results.apply(lambda x: x[0])
    df[f"{topic}_matched"] = results.apply(lambda x: x[1])

topic_score_cols = [f"{topic}_score" for topic in TOPIC_PHRASES]

df["max_topic_score"] = df[topic_score_cols].max(axis=1)

def detected_topics(row):
    return [
        topic
        for topic in TOPIC_PHRASES
        if row[f"{topic}_score"] > 0
    ]

df["detected_topics"] = df.apply(detected_topics, axis=1)

def filter_level(score):
    if score >= 10:
        return "strong_candidate"
    elif score >= 4:
        return "medium_candidate"
    elif score >= 1:
        return "weak_candidate"
    else:
        return "irrelevant"

df["filter_level"] = df["max_topic_score"].apply(filter_level)

filtered_df = df[df["max_topic_score"] > 0].copy()

filtered_df.head()

KeyboardInterrupt: 

In [ ]:
filtered_df

## Claude

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.0 MB/s eta 0:00:00


In [ ]:
!pip install pandarallel

  Preparing metadata (setup.py) ... done
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16674 sha256=3f898a25cf4122ae4456e49b65b1c8944b32fac2940c3454fe5e4bb5a0c59dc5
  Stored in directory: /root/.cache/pip/wheels/46/f9/0d/40c9cd74a7cb8dc8fe57e8d6c3c19e2c730449c0d3f2bf66b5
Successfully built pandarallel


In [ ]:
from rapidfuzz import fuzz
import pandas as pd
import re

In [ ]:
SOURCES = {
    "Klix": klix,
    "Avaz": avaz,
    "Buka": buka,
    "RTRS": rtrs,
    "Stav": stav,
    "RadioSarajevo": radiosarajevo,
}

In [ ]:
TITLE_COLS = ["NASLOV", "naslov", "title", "TITLE"]
SUBTITLE_COLS = ["PODNASLOV", "podnaslov", "LEAD", "lead", "subtitle"]
CONTENT_COLS = ["SADRZAJ", "SADRŽAJ", "sadrzaj", "sadržaj", "TEKST", "tekst", "content", "CONTENT", "TEXT", "text"]

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_field(row, col):
    if col is None:
        return ""
    val = row.get(col, "")
    if pd.isna(val):
        return ""
    return str(val)


In [ ]:
# ------------------------------------------------------------------
# Cirilica -> latinica (RTRS i drugi izvori mogu objavljivati na cirilici)
# ------------------------------------------------------------------
CYR2LAT = {
    'а':'a','б':'b','в':'v','г':'g','д':'d','ђ':'đ','е':'e','ж':'ž','з':'z',
    'и':'i','ј':'j','к':'k','л':'l','љ':'lj','м':'m','н':'n','њ':'nj','о':'o',
    'п':'p','р':'r','с':'s','т':'t','ћ':'ć','у':'u','ф':'f','х':'h','ц':'c',
    'ч':'č','џ':'dž','ш':'š',
}

def cyr_to_lat(text):
    return "".join(CYR2LAT.get(ch, ch) for ch in text)

def normalize_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).lower()
    s = cyr_to_lat(s)
    return s

WORD_RE = re.compile(r"[a-zčćžšđ0-9]+", re.UNICODE)

def tokenize(text):
    return WORD_RE.findall(text)


def stem_matches_token(stem, token, min_fuzzy_len=4, threshold=82):
    """Da li RIJEC IZ TEKSTA (token) odgovara KORIJENU (stem)?
    - Pokriva sve padeze: token koji POCINJE korijenom se racuna kao pogodak
      (genocid -> genocid/genocida/genocidu/genocidom/genocidski...).
    - Pokriva sitne tipfelere/nedostatak dijakritike: ako korijen ima vise
      od `min_fuzzy_len` slova, dozvoljava se fuzzy poređenje (rapidfuzz)
      sa pragom `threshold` (0-100) umjesto tacnog poklapanja.
    - Kratki korijeni (npr. "eu", "cik") NE koriste fuzzy (da ne hvataju sum),
      moraju se tacno poklopiti kao prefiks rijeci.
    """
    if not token:
        return False
    if len(token) < len(stem) - 1:
        return False
    if token.startswith(stem):
        return True
    if len(stem) <= min_fuzzy_len:
        return False
    prefix = token[: len(stem) + 1]
    return fuzz.ratio(stem, prefix) >= threshold


def slot_matches(alts, tokens):
    """Slot = lista alternativa (sinonimi / dijakritičke varijante).
    Zadovoljen je ako BAREM JEDNA alternativa odgovara BAREM JEDNOJ riječi."""
    return any(stem_matches_token(stem, t) for stem in alts for t in tokens)


def concept_matches(slots, tokens):
    """Koncept (npr. [["ratn"], ["zločin"]]) je pogođen ako je SVAKI slot
    zadovoljen NEGDJE u tekstu - bez obzira na redoslijed riječi (zbog
    fleksibilnog reda riječi u BHS jeziku) i bez obzira na padež."""
    return all(slot_matches(slot, tokens) for slot in slots)


def concept_label(slots):
    return " ".join(alts[0] for alts in slots)


In [ ]:
# -*- coding: utf-8 -*-
# Format koncepta: lista "slotova". Svaki slot je lista alternativnih KORIJENA
# riječi (sinonimi / dijakritičke varijante / ćirilica već transliterisana).
# Koncept je "pogođen" ako je SVAKI slot zadovoljen NEGDJE u tekstu
# (redoslijed riječi i padež nisu bitni - tipično za slovenske jezike).
# Korijen = riječ BEZ padežnog/gramatičkog nastavka, npr. "genocid" (ne "genocida"),
# "ratn" (ne "ratni"), "negir" (ne "negiranje") - tako jedan korijen pokriva SVE padeže.

TOPIC_CONCEPTS = {

    # ============== 1) EUROATLANTSKE INTEGRACIJE ==============
    "euroatlantske_integracije": [
        [["euroatlantsk"]],
        [["evroatlantsk"]],
        [["evropsk", "europsk"], ["integracij"]],
        [["evrointegracij"]],
        [["europrocesi", "evroprocesi"]],
        [["evropsk", "europsk"], ["unij"]],
        [["nato"]],
        [["sjevernoatlantsk"]],
        [["atlantsk"], ["savez"]],
        [["partnerstv"], ["mir"]],
        [["članstv"], ["eu", "nato"]],
        [["kandidatsk"], ["status"]],
        [["status"], ["kandidat"]],
        [["pregovor"], ["eu"]],
        [["pristupn"], ["pregovor"]],
        [["pregovaračk"], ["poglavlj"]],
        [["otvaranj"], ["pregovor"]],
        [["acquis"]],
        [["akcion"], ["plan"], ["članstv"]],  # MAP razloženo
        [["map"], ["nato"]],
        [["anp"]],
        [["godišnj"], ["nacionaln"], ["program"]],  # ANP razloženo
        [["program"], ["reform"]],
        [["reformsk"], ["agend"]],
        [["brisel", "bruxelles"]],
        [["evropsk", "europsk"], ["komisij"]],
        [["evropsk", "europsk"], ["parlament"]],
        [["evropsk", "europsk"], ["savjet"]],
        [["politik"], ["proširenj"]],
        [["proširenj"], ["eu"]],
        [["izvještaj"], ["napretk"]],
        [["vizn"], ["liberalizacij"]],
        [["šengen", "schengen"]],
        [["ipa"], ["fond"]],
        [["prioritet"], ["eu"]],
        [["pristupanj"], ["eu"]],
        [["bonsk"], ["ovlašćenj", "ovlašten"]],
        [["ohr"]],
        [["ured"], ["visok"], ["predstavnik"]],
    ],

    # ============== 2) NEGIRANJE GENOCIDA ==============
    "negiranje_genocida": [
        [["genocid"]],
        [["negir"], ["genocid"]],
        [["poric"], ["genocid"]],
        [["negacionizam", "negacionist"]],
        [["srebrenic", "srebrenič"]],
        [["potočar"]],
        [["memorijaln"], ["centar"]],
        [["ratn"], ["zločin"]],
        [["zločin"], ["čovječnost", "humanost"]],
        [["velič"], ["zločin"]],
        [["velič"], ["ratn"]],
        [["ratko"], ["mladić"]],
        [["karadžić"]],
        [["šešelj"]],
        [["haškt", "hašk"], ["tribunal", "sud"]],
        [["icty"]],
        [["mict"]],
        [["haag", "hag"]],
        [["presud"], ["genocid"]],
        [["presud"], ["haag", "hag"]],
        [["rezolucij"], ["genocid"]],
        [["rezolucij"], ["srebrenic"]],
        [["generaln"], ["skupštin"]],
        [["dan"], ["sjećanj"]],
        [["spomen"], ["obilježj", "ploč"]],
        [["revizij"], ["historij", "istorij"]],
        [["zabran"], ["negiranj", "poricanj"]],
        [["inzko"]],
        [["zakon"], ["negiranj"], ["genocid"]],
        [["krivičn"], ["genocid"]],
        [["žrtv"], ["genocid"]],
        [["pokolj"], ["srebrenic"]],
        [["masakr"], ["srebrenic"]],
    ],

    # ============== 3) GRAĐANSKA DRŽAVA vs. KONSTITUTIVNI NARODI ==============
    "gradjanska_vs_konstitutivni": [
        [["konstitutivn"], ["narod"]],
        [["građansk", "gradjansk"], ["država"]],
        [["građansk", "gradjansk"], ["koncept", "princip"]],
        [["legitimn"], ["predstavljanj"]],
        [["majoriz"]],
        [["jedan"], ["čovjek"], ["glas"]],
        [["unitarn"], ["bih", "bosn"]],
        [["unitarizacij", "unitarizam"]],
        [["federalizacij"]],
        [["treć"], ["entitet"]],
        [["entitetsk"], ["veto", "glasanj"]],
        [["dejtonsk"]],
        [["dejton"], ["sporazum"]],
        [["sejdić"], ["finci"]],
        [["presud"], ["ljubić"]],
        [["presud"], ["pilav"]],
        [["ustavn"], ["sud"], ["bih", "bosn"]],
        [["ustavn"], ["amandman", "promjen", "reform"]],
        [["vital"], ["nacionaln"], ["interes"]],
        [["nacionaln"], ["ključ", "kvot"]],
        [["etničk"], ["predstavljanj", "kvot"]],
        [["komšić"]],
        [["bećirović"]],
        [["džaferović"]],
        [["hrvatsk"], ["pitanj"]],
        [["bošnjačk"], ["pitanj"]],
        [["srpsk"], ["pitanj"]],
        [["bošnjačk"], ["većin"]],
        [["tač"], ["narod", "konstitutivn"]],  # tač - lider SDA, kontekst zaštitno
        [["čović"]],
        [["dodik"], ["secesij", "otcjepljenj", "republik"]],
        [["nezavisn"], ["republik"], ["srpsk"]],
        [["amandman"], ["ustav"]],
    ],

    # ============== 4) IZBORNA REFORMA ==============
    "izborna_reforma": [
        [["izborn"], ["reform"]],
        [["izborn"], ["zakon"]],
        [["zakon"], ["izbor"]],
        [["izmjen"], ["izborn"]],
        [["centraln"], ["izborn"], ["komisij"]],
        [["cik"]],
        [["dom"], ["narod"]],
        [["predstavničk"], ["dom"]],
        [["predsjedništv"], ["bih", "bosn"]],
        [["hrvatsk"], ["član"], ["predsjedništv"]],
        [["schmidt", "šmit"]],
        [["visok"], ["predstavnik"]],
        [["nametnut"], ["izmjen", "zakon", "rješenj"]],
        [["paket"], ["integritet"]],
        [["izborn"], ["integritet"]],
        [["elektronsk"], ["glasanj"]],
        [["skener"]],
        [["brojanj"], ["glas"]],
        [["izborn"], ["prag"]],
        [["hdz"]],
        [["hns"]],
        [["čović"]],
        [["legitimn"], ["predstavljanj"]],
        [["amandman"], ["izborn"]],
        [["venecijansk"], ["komisij"]],
        [["raspodjel"], ["mandat"]],
        [["delegat"], ["dom"], ["narod"]],
        [["opšt", "opć"], ["izbor"]],
        [["lokaln"], ["izbor"]],
        [["parlamentarn"], ["izbor"]],
        [["opštinsk", "općinsk"], ["izbor"]],
        [["izborn"], ["jedinic"]],
        [["bonsk"], ["ovlašćenj", "ovlašten"]],
        [["ohr"]],
    ],
}


In [ ]:
def topic_score(row, concepts, title_col, subtitle_col, content_col,
                 w_title=5, w_subtitle=3, w_content=1):
    """Vraca (ukupan_skor, overlap, lista_pogodjenih_koncepata).

    overlap = udio jedinstvenih koncepata teme koji su pogodjeni
    (broj_pogodjenih_koncepata / ukupan_broj_koncepata_teme) - ovo je
    trazeni "overlap" pokazatelj koji ne zavisi od egzaktnog poklapanja
    nego od toga koliko se SADRZAJ TEME poklapa sa clankom.
    """
    title = tokenize(normalize_text(get_field(row, title_col)))
    subtitle = tokenize(normalize_text(get_field(row, subtitle_col)))
    content = tokenize(normalize_text(get_field(row, content_col)))

    score = 0
    matched = []

    for slots in concepts:
        label = concept_label(slots)
        hit = False
        if concept_matches(slots, title):
            score += w_title
            hit = True
        if subtitle and concept_matches(slots, subtitle):
            score += w_subtitle
            hit = True
        if concept_matches(slots, content):
            score += w_content
            hit = True
        if hit:
            matched.append(label)

    overlap = len(matched) / len(concepts) if concepts else 0
    return score, round(overlap, 3), matched


In [ ]:
def apply_topics(df, source_name):
    df = df.copy()
    title_col = first_existing_col(df, TITLE_COLS)
    subtitle_col = first_existing_col(df, SUBTITLE_COLS)
    content_col = first_existing_col(df, CONTENT_COLS)

    print(f"{source_name}: naslov='{title_col}', podnaslov='{subtitle_col}', sadrzaj='{content_col}'")

    for topic, concepts in TOPIC_CONCEPTS.items():
        results = df.parallel_apply(
            lambda row: topic_score(row, concepts, title_col, subtitle_col, content_col),
            axis=1,
        )
        df[f"{topic}_score"] = results.apply(lambda r: r[0])
        df[f"{topic}_overlap"] = results.apply(lambda r: r[1])
        df[f"{topic}_matched"] = results.apply(lambda r: r[2])

    df["IZVOR"] = source_name
    return df
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
scored = {name: apply_topics(df, name) for name, df in SOURCES.items()}


INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Klix: naslov='NASLOV', podnaslov='None', sadrzaj='SADRZAJ'


In [ ]:
MIN_SCORE = 1      # bar jedan pogodak (po pondernom skoru)
MIN_OVERLAP = 0.0  # 0 = ne filtriraj po overlap-u; npr. 0.1 = bar 10% koncepata teme pogođeno

topic_score_cols = [f"{t}_score" for t in TOPIC_CONCEPTS]

filtered = {}
for name, df in scored.items():
    df["max_topic_score"] = df[topic_score_cols].max(axis=1)
    mask = df["max_topic_score"] >= MIN_SCORE
    if MIN_OVERLAP > 0:
        overlap_cols = [f"{t}_overlap" for t in TOPIC_CONCEPTS]
        mask = mask & (df[overlap_cols].max(axis=1) >= MIN_OVERLAP)
    filtered[name] = df[mask].copy()
    print(f"{name}: {mask.sum()} / {len(df)} clanaka prosli filter")


In [ ]:
combined = pd.concat(filtered.values(), ignore_index=True)
combined


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/CDA-MLPR data/filtrirano", exist_ok=True)

combined.to_csv("/content/drive/MyDrive/CDA-MLPR data/filtrirano/svi_clanci_filtrirano.csv", index=False)

for topic in TOPIC_CONCEPTS:
    subset = combined[combined[f"{topic}_score"] > 0]
    subset.to_csv(f"/content/drive/MyDrive/CDA-MLPR data/filtrirano/{topic}.csv", index=False)
    print(f"{topic}: {len(subset)} clanaka")


euroatlantske_integracije: 113518 clanaka
negiranje_genocida: 83991 clanaka
gradjanska_vs_konstitutivni: 157320 clanaka
izborna_reforma: 174128 clanaka
